# 04 - Risk Prediction Baselines and LazyPredict

## Problem framing
We model `Recovery_Status` as a multi-class prediction task and derive high-risk borrowers from the written-off class.

## Why LazyPredict?
- Fast baseline exploration
- Useful for rapid model sanity checks
- Lower control than hand-crafted pipelines


## Definition
Risk prediction estimates recovery outcomes and supports high-risk early warning decisions.

## Theory
This section explains the statistical or ML theory behind the technique and why it is useful in credit recovery operations.

## Mathematical Intuition
We translate the idea into score/probability/ranking logic so each metric can be interpreted quantitatively.

## Financial Intuition
We connect the method to borrower affordability, delinquency severity, collateral protection, and expected recoverable cashflows.

## Business Impact
We explain what this enables for collection managers, risk teams, and executives.

## Real-World Example
Two borrowers with similar outstanding balances can receive different actions if their written-off probability diverges.

## Visual Explanation
Charts in this notebook show how model/segment behavior changes across borrower groups.

## Code Explanation
Every code cell below is paired with interpretation so beginners can connect implementation details to business outcomes.

## Interpretation of Results
We summarize what changed, why it matters, and how to act on it.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.loan_recovery import (
    DATA_PATH,
    FIGURES_DIR,
    MODELS_DIR,
    TABLES_DIR,
    REPORTS_DIR,
    LoanDataLoader,
    FeatureEngineer,
    LoanEDA,
    BorrowerSegmenter,
    ModelTrainer,
    ModelEvaluator,
    RecoveryStrategyEngine,
    ModelExplainer,
    DashboardBuilder,
    LazyPredictBenchmark,
    PyCaretWorkflow,
    FLAMLOptimizer,
    SmartLoanRecoveryPipeline,
    load_model,
)

sns.set_theme(style="whitegrid")


In [ ]:
def ensure_pipeline_artifacts() -> None:
    required = [
        TABLES_DIR / "manual_model_leaderboard.csv",
        TABLES_DIR / "feature_enriched_data.csv",
        MODELS_DIR / "best_manual_model.joblib",
    ]
    if not all(path.exists() for path in required):
        pipeline = SmartLoanRecoveryPipeline(data_path=DATA_PATH, random_state=42)
        pipeline.run()

ensure_pipeline_artifacts()


In [ ]:
df = LoanDataLoader(DATA_PATH).load()
fe = FeatureEngineer()
enriched = fe.engineer(df)
split = fe.train_test_split(enriched, target_col="Recovery_Status", drop_cols=["Borrower_ID"])

trainer = ModelTrainer(random_state=42)
manual = trainer.train_baselines(split.x_train, split.y_train, split.x_test, split.y_test)
display(manual.leaderboard)


In [ ]:
x_train_lazy = pd.get_dummies(split.x_train)
x_test_lazy = pd.get_dummies(split.x_test)
x_train_lazy, x_test_lazy = x_train_lazy.align(x_test_lazy, axis=1, join="left", fill_value=0)

lazy = LazyPredictBenchmark(random_state=42)
lazy_df = lazy.run(x_train_lazy, split.y_train, x_test_lazy, split.y_test)
display(lazy_df)
display(lazy.required_model_snapshot())


In [ ]:
manual_top = manual.leaderboard[["model", "accuracy", "f1_weighted", "roc_auc_ovr"]].copy()
manual_top["source"] = "Manual"

lazy_top = lazy_df[["model", "Accuracy", "F1 Score"]].copy()
lazy_top.columns = ["model", "accuracy", "f1_weighted"]
lazy_top["roc_auc_ovr"] = np.nan
lazy_top["source"] = "LazyPredict"

comparison = pd.concat([manual_top, lazy_top], ignore_index=True)
display(comparison.sort_values(["source", "f1_weighted"], ascending=[True, False]))


## Tradeoff Summary
- **Manual ML**: highest control and deployment clarity.
- **LazyPredict**: strongest for quick benchmark discovery.
- **Weakness**: less configurable preprocessing and less business-specific tuning.
